# PaiNN C-GAIMMD on alanine dipeptide (NANMA) in vacuum

This tutorial runs the production workflow of C-GAIMMD with the PaiNN
flow-matching generator on N-acetyl-N'-methylalaninamide (alanine dipeptide,
NANMA) in vacuum, with GROMACS. The transition is the isomerization between
the C7eq (A) and C7ax (B) conformations, defined as discs in the
($\phi$, $\psi$) plane.

The steps are those of any C-GAIMMD campaign:

1. the system folder and the generator atoms;
2. unbiased MD of both end states;
3. the network and its end-state pretraining (the start of every training round);
4. the params file;
5. the preflight;
6. a local launch, and the job script for a cluster;
7. monitoring: logs, network metadata, sidecars and the shot table;
8. restarts and post-training;
9. rates through AIMMD's reweighting, and the committor checked against
   brute-force shooting.

The toy campaign of tutorial 3 explains the C-GAIMMD loop itself; this
tutorial shows what changes for a molecule.

> **Outputs pending.** This tutorial needs a GPU and GROMACS. The page
> shows the code and the explanations; the outputs of a run on a GPU
> workstation will be added. Every path that depends on your machine is set in
> the first code cell.

## 0. Paths and settings

The system files come from the NANMA test system of the AIMMD-Lab repository
(`tests/cgaimmd_nanma`: CHARMM22 without CMAP, an 8 nm box, `run.mdp` with a
1 fs time step and frames every 50 fs, `em.mdp`, and `states.py` with the
state definition). Point `SYSTEM` at your own folder with the same files to
run another molecule. The run folder belongs on a local disk.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import aimmd
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import torch

# --- configure ---------------------------------------------------------------------
AIMMD_LAB = Path(aimmd.__file__).resolve().parents[1]     # a clone of AIMMD-Lab (editable install)
SYSTEM = AIMMD_LAB / "tests" / "cgaimmd_nanma"            # conf.gro, topol.top, run.mdp, em.mdp, states.py
CAMPAIGN = Path("nanma_cgaimmd").resolve()               # params, pretrained network, end-state MD
RUN = CAMPAIGN / "run"                                    # the run folder (a local disk is best)
GMX = shutil.which("gmx") or "/path/to/gromacs/bin/gmx"   # full path: workers do not run your login shell
THREADS = 8                                               # OpenMP threads per GROMACS run
END_STATE_NS = 5.0                                        # unbiased MD per end state, ns
# ------------------------------------------------------------------------------------

CAMPAIGN.mkdir(parents=True, exist_ok=True)
for name in ("conf.gro", "topol.top", "run.mdp", "em.mdp", "states.py"):
    shutil.copy(SYSTEM / name, CAMPAIGN / name)
os.chdir(CAMPAIGN)
sys.path.insert(0, str(CAMPAIGN))
from states import states_function                        # noqa: E402

print("GROMACS:", GMX, "| CUDA available:", torch.cuda.is_available())

## 1. The generator atoms

The generator atoms are the atoms the network generates, in a fixed order. In
vacuum they are the whole molecule (22 atoms). `GeneratorDescriptors` turns
any trajectory of the system into their flat Ångström coordinates, with the
molecule made whole across the periodic box; under C-GAIMMD these coordinates
are the run's descriptors.

In [ ]:
from aimmd.cgaimmd.preparation.descriptors import GeneratorAtoms, GeneratorDescriptors
from aimmd.cgaimmd.preparation.topology import read_masses

N_ATOMS = mda.Universe("conf.gro").atoms.n_atoms
DESCRIPTORS = GeneratorDescriptors("conf.gro", GeneratorAtoms(n_solute=N_ATOMS),
                                   masses=read_masses("topol.top"))
print(N_ATOMS, "generator atoms")

## 2. Unbiased MD of the end states

The initial network is trained on unbiased MD of the two end states: before
the first shot there are no committor labels, so the network can only learn
the two basins. The frames must be plain MD (no umbrella sampling,
metadynamics or steered frames), because the generator learns the
distribution of what it is given.

A starts from `conf.gro` (C7eq); B starts from the last frame of the test
system's free MD in C7ax. The same trajectories, thinned, seed the free
simulations of the run (`initial_paths`) and serve as clash references of the
validity check.

In [ ]:
u = mda.Universe("conf.gro", str(SYSTEM / "free_md_B.xtc"))
u.trajectory[-1]
u.atoms.write("conf_B.gro")


def end_state_md(name, structure, ns):
    """Plain GROMACS MD of one end state (run.mdp has a 1 fs time step)."""
    subprocess.run([GMX, "grompp", "-f", "run.mdp", "-c", structure, "-p", "topol.top",
                    "-o", f"{name}.tpr", "-maxwarn", "5"], check=True, capture_output=True)
    subprocess.run([GMX, "mdrun", "-deffnm", name, "-nsteps", str(int(ns * 1e6)),
                    "-ntmpi", "1", "-ntomp", str(THREADS), "-nb", "cpu"],
                   check=True, capture_output=True)


for name, structure in (("free_A", "conf.gro"), ("free_B", "conf_B.gro")):
    end_state_md(name, structure, END_STATE_NS)

# every 100th frame as the per-state seed files of the run
for name in ("A", "B"):
    u = mda.Universe("conf.gro", f"free_{name}.xtc")
    with mda.Writer(f"initial_{name}.xtc", u.atoms.n_atoms) as writer:
        for ts in u.trajectory[::100]:
            writer.write(u.atoms)
    labels = states_function(mda.Universe("conf.gro", f"initial_{name}.xtc").trajectory)
    print(f"initial_{name}.xtc:", dict(zip(*np.unique(labels, return_counts=True))))

## 3. The network

`JointPaiNN.from_topology` builds the PaiNN network from the GROMACS topology:
atom tokens, bond edges and the permutation symmetries of the molecule. Every
architecture setting must be given; the values below are those used for NANMA.
The network is SE(3)-equivariant (`parity_breaking=True`), so it can tell the
L-alanine of NANMA from its mirror image, and it sees the topology's bonded
structure at every flow time (`topology_edges=True`).

`coord_scale` sets how wide the Gaussian prior is in Ångström; measure it once
on the end-state frames and keep it, because changing it invalidates every
checkpoint.

In [ ]:
from aimmd.cgaimmd.flow_matching.pretrain import collect_end_state_frames, estimate_coord_scale

frames = collect_end_state_frames(["free_A.xtc", "free_B.xtc"], structure="conf.gro",
                                  descriptors_function=DESCRIPTORS,
                                  states_function=states_function, states="ARB", stride=10)
COORD_SCALE = estimate_coord_scale(frames["coordinates"])
print(len(frames["state"]), "end-state frames:", dict(zip(*np.unique(frames["state"], return_counts=True))))
print(f"coord_scale = {COORD_SCALE:.4f} A")

## 4. The params file

The params file is written in two steps: first without `initial_network`, so
that it can be loaded to build the network the pretraining trains; then with
it. Its parts:

- the GROMACS engine with the executables by full path and the thread count;
- `chain_type = 'cgaimmd'`, per-state seed files, `round_shooting_index` and
  `masses_from_topology` (CHARMM names some carbons `CL`, which would read as
  chlorine);
- `cgaimmd_generator`: the ODE with the midpoint rule, 16 steps;
- `cgaimmd_parameters`: 32 targets per refill and a fixed seed; the final
  check keeps its default rule, `'transition_region'`: a candidate is shot if
  it lies in the transition region by the state definition, whatever the
  network's committor (which is recorded with the shot);
- the descriptors, the network (a subclass defined in the params file);
- `TopologyValidity` as the validity check (bond graph, relabelling, the
  stereocentre of the alanine, clashes against the unbiased end-state MD), no
  preparation function (the molecule is the whole system and is placed in the
  box), a restrained minimization;
- the flow-matching fit: 4,000 steps per round, labels taken again every
  1,000 steps.

In [ ]:
PARAMS = f'''
"""PaiNN C-GAIMMD on NANMA in vacuum (tutorial 4)."""
import numpy as np

from aimmd.cgaimmd.flow_matching.fit import flow_matching_fit
from aimmd.cgaimmd.flow_matching.painn import JointPaiNN
from aimmd.cgaimmd.preparation.descriptors import GeneratorAtoms, GeneratorDescriptors
from aimmd.cgaimmd.preparation.minimize import GromacsMinimizer
from aimmd.cgaimmd.preparation.topology import read_masses
from aimmd.cgaimmd.preparation.validity import TopologyValidity
from states import states_function

GMX = {GMX!r}
engine = 'gromacs'
chain_type = 'cgaimmd'
states = 'ARB'
topology = 'conf.gro'
gmx_mdp = 'run.mdp'
gmx_grompp = f'{{GMX}} grompp -maxwarn 5 -p topol.top'
gmx_mdrun = f'{{GMX}} mdrun -ntmpi 1 -ntomp {THREADS} -nb cpu -pin off'
initial_paths = ['initial_A.xtc', 'initial_B.xtc']
round_shooting_index = True
masses_from_topology = True
gen_temperature = 300.0

cgaimmd_generator = {{'method': 'ode', 'sampler': 'midpoint', 'n_steps': 16, 'chunk': 128}}
cgaimmd_parameters = {{'n_targets': 32, 'generation_seed': 1}}

DESCRIPTORS = GeneratorDescriptors('conf.gro', GeneratorAtoms(n_solute={N_ATOMS}),
                                   masses=read_masses('topol.top'))


def descriptors_function(trajectory):
    return DESCRIPTORS(trajectory)


class Network(JointPaiNN):
    pass


network = Network.from_topology(
    'topol.top', DESCRIPTORS, embedding='element', topology_edges=True, parity_breaking=True,
    coord_scale={COORD_SCALE:.4f}, lambda_clip=10.0, n_channels=128, n_layers=4, n_basis=20,
    cutoff=5.0, cond_dim=64, lambda_embedding_scale=4.0, committor_hidden=(128,),
    velocity_hidden=128, velocity_radial=32, velocity_min_distance=0.7, edge_backend='torch')

_validity = TopologyValidity('topol.top', DESCRIPTORS,
                             clash_reference=['free_A.xtc', 'free_B.xtc'])


def cgaimmd_validity_function(positions, context):
    return _validity(positions, context)


_minimizer = GromacsMinimizer(em_mdp='em.mdp', restrain='generator_heavy', restraint_k=1e4,
                              emtol=100.0, nsteps=1000)


def cgaimmd_minimize_function(frame, context):
    return _minimizer(frame, context)


def fit(params, pathensemble, verbose=False, worker=None):
    return flow_matching_fit(params, pathensemble, verbose=verbose, worker=worker,
                             steps_per_round=4000, relabel_every=1000)
'''
Path("params.py").write_text(PARAMS.lstrip())
params = aimmd.Params.load("params.py", save=False)
print(type(params.network).__name__, "with", sum(p.numel() for p in params.network.parameters()),
      "parameters, generating", params.network.n_atoms, "atoms")

### The initial network: end-state pretraining

`pretrain_end_states` trains the network of the params file on the end-state
frames: the generator learns the two basins (flow matching conditioned at
$\lambda = \mp L$) and the committor separates them. The result is written as
`pretrained/networkARB.h5`, and the params file names it as
`initial_network`: every training round of the run starts again from it. On a
GPU, 30,000 steps take minutes.

In [ ]:
from aimmd.cgaimmd.flow_matching.pretrain import pretrain_end_states

report = pretrain_end_states(params.network, frames, "pretrained", states="ARB",
                             steps=30000, device="auto", seed=0)
print({key: report[key] for key in ("network_file", "steps") if key in report})

with open("params.py", "a") as handle:
    handle.write("\ninitial_network = 'pretrained/networkARB.h5'\n")
params = aimmd.Params.load("params.py", save=False)
print("initial_network:", params.initial_network)

## 5. The preflight

The preflight runs what a job runs first: it loads the params, checks the
files, the fit's keywords, the generator and the numbers, generates two
candidates and takes the first valid one through the validity check, the
minimization and the final check, and checks the GROMACS executables. Exit
status 0 means the run can start.

In [ ]:
result = subprocess.run([sys.executable, "-m", "aimmd.cgaimmd.preflight", "params.py",
                         "--candidates", "2"], capture_output=True, text=True)
print(result.stdout)
print("exit status:", result.returncode)

## 6. Launch

### Locally

The Launcher starts every worker of the run at once: here two shooting chains,
one free simulation per state and the trainer, all on this machine. It writes
the initial network into the run folder before anything else. The run stops
after 400 shots or the walltime, whichever comes first; free simulations run
until the walltime.

In [ ]:
launcher = aimmd.Launcher(params, str(RUN))
launcher.run(2, 1, 1, nsteps=400, walltime=6 * 3600)

### As a cluster job

`create_job` writes a SLURM job script for the same run: the SLURM header
(account and partition come from `params.slurm_header`), the `preamble` lines
(modules, the environment), the run-folder guard, the preflight, one `srun`
step per worker. Write it on the cluster, after loading the job's modules and
environment: AIMMD writes the interpreter, the worker script and the GROMACS
executables it finds into the script.

In [ ]:
launcher.create_job(
    "job.sh", 4, 1, 1, walltime=12 * 3600, ntasks_per_node=4, cpus_per_task=THREADS,
    gpus_per_task=1, skip_binding=False,
    preamble=["module load GROMACS",                      # your site's module lines
              "source /path/to/cgaimmd_venv/bin/activate"],
    preflight=True)
print(Path("job.sh").read_text())

## 7. Monitoring

The shooting worker's log has one line per pool refill and one per shooting
point; the trainer's log has the rounds and the rate estimates;
`networkARB.json` describes the network the workers use. The sidecars hold
everything about each shot, and `shot_table` turns them into rows without
writing into the run folder.

In [ ]:
for line in open(RUN / "chainR0" / "worker.log"):
    if line.startswith(("C-GAIMMD", "WARNING")):
        print(line.rstrip())
print(json.dumps({key: json.load(open(RUN / "networkARB.json"))[key]
                  for key in ("round", "n_trained_shots", "status", "seed")}, indent=1))

In [ ]:
from aimmd.cgaimmd.sidecar import shot_table, summarise_shots

rows = shot_table(RUN, params=params)
summary = summarise_shots(rows, frame_interval_ps=0.05)
print("shots:", summary["n_shots"], "| reactive:", summary["n_reactive"],
      "| rules:", summary["acceptance_rules"])
print("rejected in stage 2:", summary["candidates_rejected"])
print("rejected in the refills:", summary["stage1_rejected"])

target = np.array([r["lambda_target"] for r in rows if r["lambda_target"] is not None])
final = np.array([r["lambda_final"] for r in rows if r["lambda_target"] is not None])
fig, ax = plt.subplots(figsize=(4.8, 4.2), dpi=110)
ax.plot([-3, 3], [-3, 3], color="#52514e", linestyle="--", linewidth=1)
ax.scatter(target, final, s=24, color="#2a78d6")
ax.set_xlabel(r"target committor $\lambda^*$ (logit)")
ax.set_ylabel(r"committor of the shot frame $\lambda$ (logit)")
ax.set_title("Every shot: target against final committor", fontsize=10)
plt.show()

## 8. Restarts and post-training

A stopped run continues with the same command: call `launcher.run(...)` again,
or resubmit `job.sh`. GROMACS segments resume from their checkpoints, an
interrupted shot keeps its in-flight record, and the trainer continues its
round count from `networkARB.json`; every round still starts from the
pretrained network.

To train more rounds on the data of the run without changing it, mirror it into
a new folder and run only a trainer there:

In [ ]:
from aimmd.cgaimmd.restart import derive_run

report = derive_run(str(RUN), str(CAMPAIGN / "posttrain"), params=params)
aimmd.Worker(params, str(CAMPAIGN / "posttrain")).train(nrounds=2)
print(json.load(open(CAMPAIGN / "posttrain" / "networkARB.json"))["round"])

## 9. Rates and the committor against brute-force shooting

### Rates

The shots of a C-GAIMMD run are reweighted like RFPS shots, so the trainer's
rate estimates come from AIMMD's own reweighting. The trainer prints them in
every round; `extract_rate_estimates_from_log_file` reads them. Estimators that
need the probability with which a configuration becomes a shooting point (exact
path importance sampling, path MBAR) do not apply to C-GAIMMD, because that law
is not known in closed form.

In [ ]:
from aimmd.analysis.utils import extract_rate_estimates_from_log_file

t, k12, k21 = extract_rate_estimates_from_log_file(str(RUN / "trainARB.log"))
dt_ps = 0.05                                       # time between saved frames
fig, ax = plt.subplots(figsize=(5.5, 3.8), dpi=110)
ax.plot(t * dt_ps / 1000, k12 / dt_ps * 1e3, color="#2a78d6", label="A to B")
ax.plot(t * dt_ps / 1000, k21 / dt_ps * 1e3, color="#eb6834", label="B to A")
ax.set_yscale("log")
ax.set_xlabel("simulated time in the path ensemble (ns)")
ax.set_ylabel("rate estimate (1/ns)")
ax.set_title("Rate estimates of the trainer, round by round", fontsize=10)
ax.legend(frameon=False)
plt.show()

### The committor against brute-force shooting

The learned committor is checked on shooting points the run generated, spread
over the committor range: AIMMD's sweep mode shoots unbiased two-way
trajectories from each frame many times, and the share that reaches B first is
the brute-force committor of that frame. The sweep runs as a plain RFPS sweep
of the same system, with the frames as its initial path.

In [ ]:
# up to 40 registered shooting frames, spread over the committor range
chosen = sorted((r for r in rows if r["lambda_registered"] is not None),
                key=lambda r: r["lambda_registered"])
chosen = [chosen[i] for i in np.linspace(0, len(chosen) - 1, min(40, len(chosen))).astype(int)]
u = mda.Universe("conf.gro")
with mda.Writer("validation.xtc", u.atoms.n_atoms) as writer:
    for row in chosen:
        path = mda.Universe("conf.gro", str(RUN / row["chain"] / row["path"]))
        path.trajectory[row["shooting_index"]]
        writer.write(path.atoms)

sweep = aimmd.Params.load("params.py", save=False)
sweep.chain_type = "rfps"                          # a plain sweep, no generation
sweep.initial_paths = ["validation.xtc"]
n_frames = len(chosen)
aimmd.Launcher(sweep, "validation").run(4, reactive_region_mode="sweep", nsteps=10 * n_frames)

counts = np.zeros((n_frames, 2))
for shots in sweep.shot_paths("validation", "sweep"):
    counts += shots.shooting_results(sweep_size=n_frames)
brute_force = counts[:, 1] / counts.sum(axis=1)

# the committor of the run's last network on the same frames
params.update_network(str(RUN))
x = DESCRIPTORS(mda.Universe("conf.gro", "validation.xtc").trajectory)
model = 1 / (1 + np.exp(-np.asarray(params.values_function(x), dtype=float).ravel()))

fig, ax = plt.subplots(figsize=(4.6, 4.4), dpi=110)
ax.plot([0, 1], [0, 1], color="#52514e", linestyle="--", linewidth=1)
ax.errorbar(model, brute_force, yerr=np.sqrt(brute_force * (1 - brute_force) / counts.sum(axis=1)),
            fmt="o", color="#2a78d6", markersize=5)
ax.set_xlabel("committor of the network")
ax.set_ylabel("committor from brute-force shooting")
ax.set_title("Learned committor against brute force (binomial errors)", fontsize=10)
plt.show()

## Where to go from here

- {doc}`../cgaimmd` explains every setting used here; {doc}`../cgaimmd_production`
  covers installs, job scripts, restarts and backups on clusters.
- Tutorials 5 and 6 run C-GAIMMD with BioEmu2 as the generator, for a protein
  and for a protein-ligand complex.